# <center>C5: Deep Learning Deep Dive and Concepts</center>

## 1. The AI Landscape

It is important to understand the hierarchy of these terms:

-   **Artificial Intelligence (AI)**: The broad field of creating smart machines capable of performing tasks that typically require human intelligence.
-   **Machine Learning (ML)**: A subset of AI. It gives computers the ability to learn without being explicitly programmed.
-   **Deep Learning (DL)**: A specialized subset of ML. It relies on Artificial Neural Networks (ANNs) with multiple hidden layers to learn from vast amounts of data.

![AI vs ML vs DL Diagram](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fmiro.medium.com%2Fv2%2Fresize%3Afit%3A1200%2F0*ScFv33VKYkqmqGXD.png&f=1&nofb=1&ipt=f2698b25de6daf5a0a9efe43d0d4f32b333bca777bbaef669bb4f78e551bcc61)

*Relationship between AI, ML, and DL (Source: Wikimedia Commons)*

## 2. Why Deep Learning Now?

Deep Learning has existed theoretically for decades, but it has exploded in popularity recently due to three main factors:
1.  **Big Data**: We have massive amounts of labeled data (images, text, logs) to train these hungry models.
2.  **Hardware (GPUs)**: Graphics Processing Units (GPUs) are incredibly efficient at the matrix multiplications required for neural networks.
3.  **Algorithmic Innovations**: Improvements like ReLU activation, Better Initialization (He/Glorot), and Optimizers (Adam) made training deep networks feasible.

### Feature Learning vs. Feature Engineering
-   **Classical ML**: Requires manual "Feature Engineering" (e.g., detecting edges in an image manually).
-   **Deep Learning**: Performs "Feature Learning" (Representation Learning). The network learns hierarchical representations—edges in the first layer, shapes in the next, and objects in the deeper layers.

## 3. Challenges in Training Deep Networks

Training a network with many layers is difficult. Two primary challenges are the **Vanishing Gradient Problem** and **Overfitting**.

![Vanishing Gradient Problem](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fmachinemindscape.com%2Fwp-content%2Fuploads%2F2024%2F02%2FVanishing-Gradient.png&f=1&nofb=1&ipt=7a4e3a03df61bff05261b88a2f536c065a3c4c4b0aba3165fcf301251ec7407a)

![Underfitting and Overfitting](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fmachinemindscape.com%2Fwp-content%2Fuploads%2F2024%2F03%2FModel-Underfitting-Overfitting-and-Right-Fit.png&f=1&nofb=1&ipt=c07478eec9121fa5b4f734d66dac9beba38807e0a5b117b7f73e984d02af968b)

### 3.1 The Vanishing Gradient Problem

During Backpropagation, gradients are propagated from the output layer back to the input layers using the **Chain Rule**. If we use activation functions like **Sigmoid** or **Tanh**, their derivatives are always less than 1 (max 0.25 for Sigmoid). 

When you multiply many small numbers together (e.g., $0.25 \times 0.25 \times 0.25 ...$), the result approaches zero. This means early layers stop learning.

**Solution**: use **ReLU (Rectified Linear Unit)**.
-   ReLU: $f(x) = max(0, x)$
-   Derivative: 1 (if $x > 0$)
-   Multiplying 1s preserves the gradient!

Let's visualize this.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

x_vals = np.linspace(-5, 5, 100)

def sigmoid_grad(x):
    s = 1 / (1 + np.exp(-x))
    return s * (1 - s)

def relu_grad(x):
    return np.where(x > 0, 1.0, 0.0)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.plot(x_vals, sigmoid_grad(x_vals), label='Sigmoid Derivative')
plt.fill_between(x_vals, sigmoid_grad(x_vals), alpha=0.3)
plt.title("Sigmoid Derivative (Max 0.25)")
plt.grid()

plt.subplot(1, 2, 2)
plt.plot(x_vals, relu_grad(x_vals), label='ReLU Derivative', color='orange')
plt.fill_between(x_vals, relu_grad(x_vals), alpha=0.3, color='orange')
plt.title("ReLU Derivative (1 for x>0)")
plt.grid()

plt.show()

### 3.2 Overfitting

Deep networks have millions of parameters. They can easily memorize training data, leading to **Overfitting** (Low Training Loss, High Validation Loss).

**Solutions (Regularization)**:
1.  **Dropout**: Randomly ignore a percentage of neurons during training. This forces the network to learn robust features and prevents co-adaptation.
2.  **Early Stopping**: Stop training when validation loss stops decreasing.
3.  **L1/L2 Regularization**: Add a penalty term to the loss function for large weights.

## 4. Optimizers

Gradient Descent is the engine of training. There are smarter variations:

-   **SGD (Stochastic Gradient Descent)**: Updates weights using small batches. Noisy but efficient.
-   **Momentum**: Accumulates past gradients to smooth updates and accelerate learning (like a ball rolling down a hill).
-   **RMSprop**: Adapts learning rate for each parameter.
-   **Adam (Adaptive Moment Estimation)**: Combines Momentum and RMSprop. It's the most popular default choice today.

Let's compare them on a dataset.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras import Input
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD, Adam, RMSprop

# 1. Create Synthetic Data (Moons dataset is non-linear)
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Function to build and train model with specific optimizer
def train_model(optimizer, name):
    model = Sequential([
        Input(shape=(2,)),
        Dense(64, activation='relu'),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    history = model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0, validation_split=0.2)
    return history

# Train with different optimizers
hist_sgd = train_model(SGD(learning_rate=0.01), 'SGD')
hist_adam = train_model(Adam(learning_rate=0.01), 'Adam')
hist_rmsprop = train_model(RMSprop(learning_rate=0.01), 'RMSprop')

# Plot Losses
plt.figure(figsize=(10, 6))
plt.plot(hist_sgd.history['val_loss'], label='SGD Val Loss')
plt.plot(hist_adam.history['val_loss'], label='Adam Val Loss')
plt.plot(hist_rmsprop.history['val_loss'], label='RMSprop Val Loss')
plt.title('Optimizer Comparison (Validation Loss)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

## 5. Regularization in Action: Dropout

Let's see how adding Dropout layers can change a model definition.

In [ ]:
model_dropout = Sequential([
    Dense(128, activation='relu', input_shape=(2,)),
    Dropout(0.5), # Drops 50% of neurons randomly during training pass
    Dense(64, activation='relu'),
    Dropout(0.3), # Drops 30% of neurons
    Dense(1, activation='sigmoid')
])

model_dropout.summary()

## 6. What's Next: Specialized Architectures

This notebook covered the fundamentals with **Dense (fully connected)** layers. For specialized data types, there are dedicated architectures:

- **CNNs (Convolutional Neural Networks)**: Best for images and spatial data
- **RNNs (Recurrent Neural Networks)**: Best for sequences (text, time series)

> **Note**: These architectures are covered in **Module 07: Specialized Architectures** after you complete the core curriculum.

---

## 7. Exercises

**Exercise 1: Overfitting Demonstration**
Create a very deep network (e.g., 5-6 layers with many neurons) on the `make_moons` dataset with a SMALL amount of data (e.g., `n_samples=100`). Train it for many epochs (e.g., 200). Plot the training vs validation loss. Do you see the curves diverge?

**Exercise 2: Fix it with Dropout**
Take the network from Exercise 1 and add Dropout layers. Retrain and compare the results.

In [ ]:
# Code for Exercise 1
